# TRPG 룰북 RAG

## ⚙️ 설정값

In [ ]:
# Ex00. 설정값 (바꿀 값은 여기만)

# ── 모델 ──
OLLAMA_URL = "http://localhost:11434"   
EMBED_MODEL = "bge-m3"                    # 임베딩 모델 (글 → 숫자 벡터)
GEN_MODEL = "qwen2.5:14b"                 # 답변 생성 모델
TRANSLATE_MODEL = GEN_MODEL           

# ── 경로 ──
DATA = "/content/drive/MyDrive/KT 기본프로젝트"
OUT_PATH = f"{DATA}/rules_new.jsonl"          # 룰북 청크 파일
EMB_PATH = f"{DATA}/rules_new_emb.npy"        # 임베딩 저장 파일
CACHE_PATH = f"{DATA}/cache_dev_14b.json"     # 번역·검색 캐시 파일

# ── 검색 ──
TOP_K = 5                                 # 최종으로 가져올 청크 수
POOL_K = 10                               # 검색 하나당 후보 수 (합치기 전)
SEARCH_WEIGHTS = [1, 1, 1, 0.5, 0.5]      # 검색 5갈래의 비중: 임베딩 3개, 키워드 2개
RRF_C = 60                                # RRF 점수 = 비중 / (RRF_C + 순위)
TRANSLATE_CTX = 4096                      # 번역 호출의 문장길이

# ── 답변 ──
GEN_SEED = 42                             # 같은 질문 → 같은 답이 나오게 고정
GEN_CTX = 8192                            # 답변 호출의 문장길이
SELECT_CHARS = 300                        # 근거 고를 때 후보마다 보여 줄 글자 수
VERIFY_CHARS = 800                        # 근거 검증 때 보여 줄 글자 수
CTX_CHARS = 2000                          # 설명 쓸 때 보여 줄 글자 수
EXPLAIN_TRIES = 3                         # 설명이 깨지면 다시 쓰는 최대 횟수
CHAT_ATTEMPTS = 3                         # 챗봇 모드에서 검사에 걸리면 다시 하는 최대 횟수
DEV_VIEW = True                           # True = 개발 화면(원문·출처) / False = 챗봇 화면(설명만)

## 0. 설치

In [ ]:
%%writefile /content/requirements_rb.txt
pymupdf==2.0
requests==2.32.4
faiss-cpu==1.15.1

In [ ]:
# 설치
!pip install -r /content/requirements_rb.txt -q
!pip freeze | grep -E "pymupdf|requests|faiss"

## 1. Ollama 설치 · 서버 · 모델 받기
모델은 **백그라운드로** 받아요. 다운로드가 도는 동안 다음 셀들을 진행해도 돼요.

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# Ex00-b. import · Ollama 서버 켜기
import os, re, json, time, math, subprocess
from collections import Counter
import numpy as np, requests, faiss
from tqdm.auto import tqdm
from google.colab import drive, files


def wait_server(url, seconds=120):     
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass                     
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. ollama.log를 확인하세요")


os.environ["OLLAMA_KEEP_ALIVE"] = "1h"    # 모델을 1시간 동안 메모리에 유지
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)   # 서버를 백그라운드로 켜기
wait_server(OLLAMA_URL)

In [ ]:
# 모델 2개(임베딩 + 14B)를 백그라운드로 받기
subprocess.Popen(f'nohup sh -c "ollama pull {EMBED_MODEL} && ollama pull {GEN_MODEL}" > pull.log 2>&1 &', shell=True)
print("다운로드 시작. 진행 확인은 아래 셀")

다운로드 시작. 진행 확인은 아래 셀


In [ ]:
# 모델이 다 받아질 때까지 기다리기
t0 = time.time()
ready = False

while time.time() - t0 < 1200:                     
    out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout   # 받은 모델 목록
    if GEN_MODEL in out and EMBED_MODEL in out:
        print(f"모델 준비 완료 ({int(time.time() - t0)}초 기다림)")
        ready = True
        break
    print("아직 받는 중...", int(time.time() - t0), "초")
    time.sleep(15)                                   # 15초마다 확인

if not ready:
    print("20분이 지나도 안 받아졌어요. pull.log를 확인하세요:")
    print(open("pull.log").read()[-500:])            # 로그 끝부분 500글자

## 2. 구글 드라이브 · 경로

In [8]:
drive.mount("/content/drive")
print(OLLAMA_URL, "| LLM(번역·생성 공통):", GEN_MODEL, "| 임베딩:", EMBED_MODEL)


Mounted at /content/drive
http://localhost:11434 | LLM(번역·생성 공통): qwen2.5:14b | 임베딩: bge-m3


## 3. 청크 읽기 · 임베딩(캐시) · FAISS 인덱스

In [ ]:
# Ex01. 저장한 청크 읽기
chunks = []
for line in open(OUT_PATH, encoding="utf-8"):   # jsonl = 한 줄에 청크 하나
    c = json.loads(line)
    if c["meta"]["type"] == "rule":              # 규칙 청크만 검색 대상
        chunks.append(c)

print("검색 대상 청크:", len(chunks))

# 검색 대상 청크: 1957

In [ ]:
# Ex02. 임베딩: 청크 글 → 숫자 벡터

def embed_batch(texts, model=EMBED_MODEL):      # 글 여러 개를 한 번에 벡터로 바꾸기
    res = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
        timeout=300,
    )
    res.raise_for_status()                       # 서버 오류면 여기서 멈춤
    return res.json()["embeddings"]


BATCH = 32                                    

# 저장된 임베딩이 있고 청크 수도 같으면 → 다시 계산하지 않고 불러오기
saved = np.load(EMB_PATH) if os.path.exists(EMB_PATH) else None

if saved is not None and saved.shape[0] == len(chunks):
    emb = saved
    print("캐시 사용:", emb.shape)
else:
    vecs = []
    for i in tqdm(range(0, len(chunks), BATCH)):
        batch_texts = [c["text"] for c in chunks[i:i + BATCH]]
        vecs += embed_batch(batch_texts)
    emb = np.array(vecs, dtype="float32")
    np.save(EMB_PATH, emb)
    print("새로 계산:", emb.shape)

# 캐시 사용: (1957, 1024)

In [ ]:
# Ex03. FAISS 인덱스 만들기
vecs = emb.copy()
faiss.normalize_L2(vecs)                    # 벡터 길이를 1로 맞춤 → 내적 = 코사인 유사도
index = faiss.IndexFlatIP(vecs.shape[1])    # IP = 내적으로 비교하는 인덱스 (벡터 차원 1024)
index.add(vecs)
print("인덱스에 들어간 벡터 수:", index.ntotal)

# 인덱스에 들어간 벡터 수: 1957

## 4. 평가 도구와 질문 세트

In [ ]:
# Ex04-a. 제목 비교: 검색된 청크 제목이 정답 제목과 같은지

# 정답을 줄여 적은 경우의 보정표: 정답이 "Grappl"이면 grappled, grappling 둘 다 정답으로 인정
ALIAS = {"Grappl": {"grappled", "grappling"}}


def norm(title):                    # 제목 다듬기 (꼬리표 제거 + 소문자)
    title = re.sub(r"\s*\[[^\]]*\]$", "", title)    # 예: "Prone [Condition]" → "Prone"
    title = re.sub(r"\s*\([^)]*\)$", "", title)     # 예: "Hide (Action)"     → "Hide"
    return title.strip().lower()


def is_hit(title, want):            # 정답인지 (정확한 비교)
    same_title = norm(title) == norm(want)
    in_alias = norm(title) in ALIAS.get(want, ())   # 보정표에 없는 정답이면 빈 묶음 → 항상 False
    return same_title or in_alias


def stem(title):                    # 단수/복수 차이 없애기 (단어 끝 s 제거)
    return re.sub(r"s\b", "", norm(title))          # 예: "attacks" → "attack"


def is_hit_loose(title, want):      # 정답인지 (느슨한 비교, 근거 선택 평가용)
    same_stem = stem(title) == stem(want)
    in_alias = norm(title) in ALIAS.get(want, ())
    return same_stem or in_alias


# 확인
print(norm("Prone [Condition]"), "|", is_hit("Prone [Condition]", "Prone"))
print(is_hit("Grappled", "Grappl"), "|", is_hit_loose("Attacks", "Attack"))

In [ ]:
# Ex04-b. 채점: 정답이 몇 등에 있나 → (1등, 3등 안, 5등 안) 개수 세기

def wants_of(g):                    # 원하는 golden(g)  # 정답을 항상 리스트로 ("Prone" → ["Prone"])
    if isinstance(g["want"], str):
        return [g["want"]]
    return g["want"]


def rank_in(found, g):              # 정답 등수 찾기 (없으면 None)
    titles = [c["meta"]["title"] for c in found]
    for i, title in enumerate(titles):
        if any(is_hit(title, w) for w in wants_of(g)):
            return i + 1, titles                    # 등수(1부터), 검색된 제목 목록
    return None, titles


def counts(ranks):                  # 순위 세기 → (1등, 3등 안, 5등 안)
    top1 = sum(1 for r in ranks if r == 1)
    top3 = sum(1 for r in ranks if r is not None and r <= 3)
    top5 = sum(1 for r in ranks if r is not None)   # 검색이 5개만 가져오므로 "찾기만 하면" 5등 안
    return (top1, top3, top5)


def evaluate(fn, items, show_miss=False):   # 질문 세트 전체 채점 (fn = 검색 함수)
    ranks = []
    for g in tqdm(items, leave=False):
        found = fn(g["q"])                          # 검색에는 질문(q)만
        rank, titles = rank_in(found, g)            # 정답(want)은 채점에만
        ranks.append(rank)
        if show_miss and rank != 1:                 # 1등 못 한 질문 출력
            print(f"[{rank}] {g['q']}\n     정답: {g['want']} | 검색: {titles[:3]}")
    return counts(ranks)


# 확인 (가짜 검색 결과로)
fake = [{"meta": {"title": t}} for t in ["Shove", "Prone [Condition]", "Grappled"]]
print(rank_in(fake, {"want": "Prone"})[0], "|", counts([1, 2, None, 5]))

# 2 | (1, 2, 3)

In [ ]:
# Ex04-c. 질문 세트 불러오기 + 정답 제목이 룰북에 실제로 있는지 확인

def load_set(name):                 # 질문 세트 파일 읽기 (없으면 빈 리스트)
    path = f"{DATA}/{name}.json"
    if not os.path.exists(path):
        return []
    return json.load(open(path, encoding="utf-8"))


golden       = load_set("golden")          # 영어 25문항
golden_ko    = load_set("golden_ko")       # 한국어 28문항
golden_new   = load_set("golden_new")      # 한국어 13문항
golden_final = load_set("golden_final")    # 최종 60문항

# 정답 제목에 오타가 있으면 영원히 0점 → 룰북 제목 목록과 대조
titles_all = {norm(c["meta"]["title"]) for c in chunks}

all_sets = {"golden": golden, "golden_ko": golden_ko,
            "golden_new": golden_new, "golden_final": golden_final}
for name, items in all_sets.items():
    miss = []
    for g in items:
        for w in wants_of(g):
            if norm(w) not in titles_all and w not in ALIAS:   # 보정표에 있는 줄임말은 통과
                miss.append(w)
    print(f"{name}: {len(items)}문항 | 룰북에 없는 정답: {miss}")

# 개발 3세트 묶음 (점수 비교용)
sets3 = {"영어": golden, "한국어ko": golden_ko, "한국어new": golden_new}

# golden: 25문항 | 룰북에 없는 정답: []
# golden_ko: 28문항 | 룰북에 없는 정답: []
# golden_new: 13문항 | 룰북에 없는 정답: []
# golden_final: 60문항 | 룰북에 없는 정답: []

## 5. 번역 (한국어 질문 → 영어 검색어)
- 번역①: 힌트 없는 일반 번역(예시 3개)
- 번역②: 규칙 용어 27개 선택지 번역
- 호출은 **14B, temperature 0, seed 42, num_ctx 4096**.

In [ ]:
# Ex05-a. 번역 호출

def ask_translate(prompt, model=None, temperature=0.0):    # 번역용 LLM 호출 (항상 같은 결과)
    options = {
        "temperature": temperature,        # 0 = 가장 확실한 답만
        "seed": 42,                        # 같은 입력 → 같은 출력
        "num_ctx": TRANSLATE_CTX,          # 4096 고정 (번역값을 따라감)
    }
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or TRANSLATE_MODEL,       # model을 안 주면 기본 번역 모델
              "messages": [{"role": "user", "content": prompt}],
              "stream": False,
              "options": options},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

In [ ]:
# Ex05-b. 번역 프롬프트 두 가지

def clean_query(s):                 # 번역 결과 다듬기 (첫 줄만, "Query:"·따옴표 제거)
    s = s.strip().split("\n")[0]                       # 첫 줄만
    s = re.sub(r"^\s*query\s*:\s*", "", s, flags=re.I) # 앞의 "Query:" 지우기 (대소문자 무시)
    return s.strip().strip("\"'")                      # 양 끝 따옴표 지우기


# ── 번역① : 일반 번역 (예시 3개를 보여 주고 따라 하게) ──
REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
Use the official rule terms that the rulebook would use.
Output only the query, nothing else.

Question: 숙련 보너스는 레벨마다 얼마야?
Query: Proficiency Bonus by character level

Question: 레벨업하려면 경험치가 얼마나 필요해?
Query: Experience Points needed to advance level

Question: 방어도는 어떻게 계산해?
Query: Armor Class calculation

Question: {question}
Query:"""


def translate1(q):                  # 일반 번역
    prompt = REWRITE_TEMPLATE.format(question=q)
    return clean_query(ask_translate(prompt))


# ── 번역② : 규칙 용어 27개 중에 맞는 게 있으면 그대로 쓰게 ──
# 상태이상 15개 + 행동 12개 (룰북 전부)
RULE_TERMS = ["Blinded", "Charmed", "Deafened", "Exhaustion", "Frightened", "Grappled", "Incapacitated",
              "Invisible", "Paralyzed", "Petrified", "Poisoned", "Prone", "Restrained", "Stunned", "Unconscious",
              "Attack action", "Dash action", "Disengage action", "Dodge action", "Help action", "Hide action",
              "Influence action", "Magic action", "Ready action", "Search action", "Study action", "Utilize action"]

REWRITE2 = """Translate the user's D&D question into a short English search query for the D&D rules.
If the question is about one of these official rule terms, use that term exactly as written.
Official terms: {terms}
Output only the query, nothing else.

Question: {q}
Query:"""


def translate2(q):                  # 규칙 용어 선택지
    terms = ", ".join(RULE_TERMS)
    prompt = REWRITE2.format(terms=terms, q=q)
    return clean_query(ask_translate(prompt))


print(translate1("겁에 질리면 뭘 못 해?"), "|", translate2("겁에 질리면 뭘 못 해?"))

# Effects of being Frightened | frightened action restrictions

## 6. 키워드 검색 BM25 (직접 구현, 새 설치 없음) -> 변경시도예정
임베딩은 **뜻**, BM25는 **글자 그대로의 단어**를 비교해서 서로 보완

In [ ]:
# Ex06. BM25 직접 만들기 (글자 그대로의 단어로 찾는 키워드 검색)

# 불용어: 너무 흔해서 검색에 도움이 안 되는 단어
STOP = set("a an the of to in on and or is are be can you your it its as at by for with that this if when how what do does".split())


def tokenize(s):                    # 글 → 단어 목록 (소문자, 영어·숫자만, 불용어 제외)
    words = re.findall(r"[a-z0-9]+", s.lower())
    return [w for w in words if w not in STOP]


class BM25:
    # 점수가 높아지는 경우: 단어가 많이 나옴(tf) + 드문 단어(idf) + 짧은 청크(len)

    def __init__(self, docs, k1=1.5, b=0.75):      # 청크들의 단어 통계 미리 계산
        self.k1 = k1                               # 빈도가 늘 때 점수가 오르는 정도
        self.b = b                                 # 길이 보정을 얼마나 할지
        self.tf = [Counter(d) for d in docs]       # 청크마다 단어별 등장 횟수
        self.len = np.array([len(d) for d in docs], dtype="float32")   # 청크마다 단어 수
        self.avg = self.len.mean()                 # 평균 단어 수

        # df = 그 단어가 나오는 청크 수 (한 청크에서 여러 번 나와도 1번으로)
        df = Counter()
        for d in docs:
            for w in set(d):
                df[w] += 1

        # idf = 드문 단어일수록 큰 값
        N = len(docs)
        self.idf = {}
        for w, n in df.items():
            self.idf[w] = math.log(1 + (N - n + 0.5) / (n + 0.5))

    def scores(self, query):                       # 질문 → 청크마다 점수
        s = np.zeros(len(self.tf), dtype="float32")
        for w in set(tokenize(query)):
            if w not in self.idf:                  # 룰북에 없는 단어는 건너뜀
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(w, 0)                   # 이 청크에서 단어가 나온 횟수
                if f:
                    length_norm = 1 - self.b + self.b * self.len[i] / self.avg   # 긴 청크일수록 큼
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * length_norm)
                    # 희귀도 × 빈도 효과 / 길이 보정
        return s


# 제목을 두 번 넣어서 제목 단어에 가중치
docs = []
for c in chunks:
    title = c["meta"]["title"]
    docs.append(tokenize(title + " " + title + " " + c["text"]))
bm25 = BM25(docs)


def bm25_search(q, k=10):           # 키워드 검색 → 점수 높은 청크 k개
    order = np.argsort(-bm25.scores(q))            # 점수 큰 순서의 번호 (-를 붙여 내림차순)
    return [chunks[i] for i in order[:k]]


print([c["meta"]["title"] for c in bm25_search("Two-Weapon Fighting", 3)])

# ['Two-Weapon Fighting', 'Great Weapon Fighting', 'Level 1: Fighting Style']

## 7. 최종 검색 함수 `retrieve` (임베딩 3갈래 + 키워드 2갈래 → 가중 RRF)
RRF: 각 검색의 순위를 `비중/(60+순위)` 점수로 바꿔 더함. 임베딩 3표(원문 질문, 번역①, 번역②)는 비중 1.0, 키워드 2표(번역①, ②)는 비중 0.5.

In [ ]:
# Ex07. 검색: 질문 하나 → 청크 5개

def dense_search(texts, k=5):       # 임베딩 검색 (글 여러 개 → 비슷한 청크 k개)
    v = np.array(embed_batch(texts), dtype="float32")
    faiss.normalize_L2(v)
    sc, ids = index.search(v, k)                   # 글마다 점수·청크 번호 k개씩

    # 같은 청크가 여러 번 나오면 가장 높은 점수만 남김
    best = {}
    for score_row, id_row in zip(sc, ids):
        for s, i in zip(score_row, id_row):
            best[i] = max(best.get(i, -1.0), float(s))

    order = sorted(best, key=best.get, reverse=True)   # 점수 높은 순
    return [chunks[i] for i in order[:k]]


def rrf_merge(lists, weights, k=TOP_K, c0=RRF_C):  # 여러 검색 결과를 순위 점수로 합치기
    # 점수 = 비중 / (c0 + 순위) → 여러 목록에서 위에 있을수록 점수가 높아짐
    score, byid = {}, {}
    for lst, w in zip(lists, weights):
        for r, c in enumerate(lst, 1):             # r = 순위 (1부터)
            score[c["id"]] = score.get(c["id"], 0) + w / (c0 + r)
            byid[c["id"]] = c

    order = sorted(score, key=score.get, reverse=True)
    return [byid[i] for i in order[:k]]


def retrieve(q, k=TOP_K):           # 최종 검색 (임베딩 3갈래 + 키워드 2갈래)
    en1 = translate1(q)                       
    en2 = translate2(q)                           

    lists = [
        dense_search([q], POOL_K),                 # 임베딩: 원문 질문
        dense_search([en1], POOL_K),           
        dense_search([en2], POOL_K),           
        bm25_search(en1, POOL_K),            
        bm25_search(en2, POOL_K),             
    ]
    return rrf_merge(lists, SEARCH_WEIGHTS, k), [q, en1, en2]   # (청크 5개, 검색에 쓴 글 3개)

In [ ]:
# 검색 개선 사항
# 비중 조정 → [1,1,1,0.5,0.5]를 다른 조합으로 변경 (BM25 0.3/0.7, 번역2 가중 상향 등)
# RRF 상수 → c0=60을 20/100으로 변경
# 후보 폭 → 목록당 10개를 20개로 확대
# 번역2 프롬프트 → "확실하지 않으면 용어 넣지 말기" + 핵심 명사 유지 + 예시 1~2개
# 번역 횟수 → 번역③(키워드만 추출) 추가
# BM25 제목 가중 → 제목 2번을 1번·3번으로 변경
# 불용어 → 목록 확대 또는 제거
# 청크 분할 → 긴 청크를 나누거나 짧은 청크를 합침
# 임베딩 입력 → 청크에 제목·경로를 붙여서 임베딩
# 리랭커 → 재도입 (이미 하락, 기각 상태)

## 8. 번역·검색 캐시 (개발셋을 빠르게 + 런타임이 초기화돼도 유지)

**왜 필요해요?** L4에서 `14B ↔ bge-m3`를 번갈아 부르면 모델이 메모리에서 교체돼서 질문당 18~20초가 걸려요. 번역과 검색 결과를 드라이브에 저장해 두면 개발셋은 질문당 약 6초로 줄어요.

**순서 (캐시 파일 이름 `cache_dev_14b.json`)**
1. 아래 첫 셀: 캐시 장치 설치 + 파일이 있으면 불러오기
2. 파일이 없거나 새 질문이 생기면 → 둘째 셀 `build_cache(...)` (번역만 먼저 → 검색만 나중에)
3. 저장은 둘째 셀 끝에서 해요. 개발셋 검색 점수(Ex08-c)는 🧪 실험 구간으로 옮겼어요

In [ ]:
# Ex08-a. 캐시 장치: 번역·검색 결과를 기억해 두기
# 같은 질문을 다시 물으면 모델을 부르지 않고 저장해 둔 결과를 바로 돌려줌. -> 개발/실험용

by_id = {c["id"]: c for c in chunks}     # 청크 id → 청크 (파일에서 불러올 때 id로 청크를 되찾으려고)

_tr_cache = {}   # 번역 기억장:  ("1", 질문) 또는 ("2", 질문)  →  영어 검색어
_sf_cache = {}   # 검색 기억장:  (질문, 개수)                 →  (청크 목록, [질문, 번역①, 번역②])


# ── 1. 원래 함수 보관 ─────────────────────────────────────────
# 셀을 두 번 돌려도 "캐시 함수를 또 캐시로 감싸는" 일이 없게, 처음 한 번만 보관.
if "_orig_tr1" not in globals():
    _orig_tr1 = translate1
    _orig_tr2 = translate2
    _orig_retrieve = retrieve


# ── 2. 같은 이름으로 캐시 버전 만들기 ─────────────────────────
# 이름이 같아서, 다른 셀(Ex07 retrieve 안의 번역 호출 등)도 자동으로 캐시 버전을 써요.
def translate1(q):                          # 번역① (캐시 버전)
    key = ("1", q)
    if key not in _tr_cache:              # 처음 보는 질문일 때만 모델 호출
        _tr_cache[key] = _orig_tr1(q)
    return _tr_cache[key]


def translate2(q):                          # 번역② (캐시 버전)
    key = ("2", q)
    if key not in _tr_cache:
        _tr_cache[key] = _orig_tr2(q)
    return _tr_cache[key]


def retrieve(q, k=TOP_K):                  # 검색 (캐시 버전)
    key = (q, k)
    if key not in _sf_cache:
        _sf_cache[key] = _orig_retrieve(q, k)
    return _sf_cache[key]


# ── 3. 드라이브에 저장 / 불러오기 ─────────────────────────────
# JSON은 튜플 키를 못 쓰니까 "1|질문", "질문|5"처럼 |로 이어 붙인 글자로 저장해요.
# 청크는 통째로 저장하지 않고 id만 저장해요 (불러올 때 by_id로 되찾음).
def save_cache():                           # 캐시를 드라이브에 저장
    tr = {}
    for (kind, q), en in _tr_cache.items():
        tr[f"{kind}|{q}"] = en

    sf = {}
    for (q, k), (found, texts) in _sf_cache.items():
        sf[f"{q}|{k}"] = {"ids": [c["id"] for c in found], "texts": texts}

    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump({"tr": tr, "sf": sf}, f, ensure_ascii=False)
    print("저장:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")


def load_cache():                           # 드라이브에서 캐시 불러오기
    if not os.path.exists(CACHE_PATH):
        print("캐시 파일 없음 → 아래 build_cache(dev)로 만들어요")
        return

    with open(CACHE_PATH, encoding="utf-8") as f:
        d = json.load(f)

    for key, en in d["tr"].items():
        kind, q = key.split("|", 1)            # 앞에서 한 번만 자름: "1|질문"
        _tr_cache[(kind, q)] = en

    for key, v in d["sf"].items():
        q, k = key.rsplit("|", 1)              # 뒤에서 한 번만 자름: "질문|5" (질문에 |가 있어도 안전)
        found = [by_id[i] for i in v["ids"]]
        _sf_cache[(q, int(k))] = (found, v["texts"])

    print("불러옴:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")


# ── 4. 모델 내리기 ───────────────────────────────────────────
# keep_alive 0 = 지금 바로 GPU 메모리에서 내림 (14B ↔ bge-m3 교체를 줄이려고)
def unload(model):                          # 모델을 GPU 메모리에서 내리기
    requests.post(f"{OLLAMA_URL}/api/generate", json={"model": model, "keep_alive": 0})


# ── 실행 ────────────────────────────────────────────────────
dev = golden + golden_new      # 개발셋 38문항
load_cache()

# 불러옴: 132 번역 / 66 검색

In [ ]:
# Ex08-b. 캐시 만들기: 1.번역 → 2.검색 / 이미 캐시된 질문은 건너뜀

def build_cache(items):             # 질문 세트의 번역·검색 결과를 미리 만들어 두기
    qs = [g["q"] for g in items]

    # 1. 번역: 번역①② 중 하나라도 없는 질문만
    todo_tr = []
    for q in qs:
        if ("1", q) not in _tr_cache or ("2", q) not in _tr_cache:
            todo_tr.append(q)

    if todo_tr:
        _orig_tr1(todo_tr[0])                      # 워밍업: 14B를 메모리에 올리기 (결과는 버림)
        for q in tqdm(todo_tr, desc="번역(14B)"):
            translate1(q)
            translate2(q)
        unload(TRANSLATE_MODEL)                    # 14B 내리기 → 임베딩 모델 자리 비우기

    # 2. 검색: 검색 결과가 없는 질문만
    todo_sf = []
    for q in qs:
        if (q, TOP_K) not in _sf_cache:
            todo_sf.append(q)

    for q in tqdm(todo_sf, desc="검색(bge-m3)"):
        retrieve(q)
    if todo_sf:
        unload(EMBED_MODEL)

    print(f"완료. 번역 새로 {len(todo_tr)}개, 검색 새로 {len(todo_sf)}개 → 캐시 번역 {len(_tr_cache)} / 검색 {len(_sf_cache)}")


build_cache(dev + golden_ko)
save_cache()

# 검색(bge-m3): 0/0 [00:00<?, ?it/s]
# 완료. 번역 새로 0개, 검색 새로 0개 → 캐시 번역 132 / 검색 66
# 저장: 132 번역 / 66 검색

## 9. 답변 생성 도구 (14B)
근거에 번호를 붙여 **번호를 먼저 고르고** 그 근거에서만 인용하게 해요. 용어표(`TERM_MAP`)는 검색용이 아니라 **답변의 한국어 표기를 통일**하는 용도예요.

In [ ]:
# Ex09-a. 용어표 (한국어 = 룰북 영어 용어) — 답변의 한국어 표기 통일용
TERM_V1 = {
    # 판정·전투
    "이점": "Advantage", "불리점": "Disadvantage", "능력 판정": "Ability Check",
    "공격 굴림": "Attack Roll", "내성 굴림": "Saving Throw", "기회 공격": "Opportunity Attack",
    "반응": "Reaction", "추가 행동": "Bonus Action", "집중": "Concentration",
    "이니셔티브": "Initiative", "저항": "Resistance", "취약": "Vulnerability",
    "엄폐": "Cover", "치명타": "Critical Hit", "숙련": "Proficiency", "전문화": "Expertise",
    "긴 휴식": "Long Rest", "짧은 휴식": "Short Rest",
    # 능력치
    "근력": "Strength", "민첩": "Dexterity", "건강": "Constitution",
    "지능": "Intelligence", "지혜": "Wisdom", "카리스마": "Charisma",
    # 피해 종류 13
    "산성": "Acid", "타격": "Bludgeoning", "냉기": "Cold", "화염": "Fire", "역장": "Force", "번개": "Lightning",
    "사령": "Necrotic", "관통": "Piercing", "독": "Poison", "정신": "Psychic", "광휘": "Radiant", "참격": "Slashing", "천둥": "Thunder",
    # 클래스 12
    "바바리안": "Barbarian", "바드": "Bard", "클레릭": "Cleric", "드루이드": "Druid", "파이터": "Fighter", "몽크": "Monk",
    "팔라딘": "Paladin", "레인저": "Ranger", "로그": "Rogue", "소서러": "Sorcerer", "워록": "Warlock", "위저드": "Wizard",
    # 주문 학파 8 + 주문 정보 칸
    "방호": "Abjuration", "소환": "Conjuration", "예지": "Divination", "환혹": "Enchantment", "방출": "Evocation",
    "환영": "Illusion", "강령": "Necromancy", "변환": "Transmutation",
    "시전 시간": "Casting Time", "사거리": "Range", "구성요소": "Components", "지속 시간": "Duration",
    "음성": "Verbal", "동작": "Somatic", "물질": "Material",
    # 룰북 용어집 -> 의미있나?
    "능력치와 수정치": "Ability Score and Modifier", "행동": "Action", "모험": "Adventure", "성향": "Alignment", "아군": "Ally",
    "효과 범위": "Area of Effect", "방어도": "Armor Class", "방어구 숙련": "Armor Training", "태도": "Attitude", "조율": "Attunement",
    "맹시": "Blindsight", "피투성이": "Bloodied", "밝은 빛": "Bright Light", "굴착 이동 속도": "Burrow Speed", "캠페인": "Campaign",
    "캔트립": "Cantrip", "운반 용량": "Carrying Capacity", "도전 지수": "Challenge Rating", "등반 속도": "Climb Speed",
    "상태": "Condition", "생물": "Creature", "생물 유형": "Creature Type", "원뿔": "Cone", "정육면체": "Cube", "원기둥": "Cylinder",
    "저주": "Curses", "D20 판정": "D20 Test", "피해": "Damage", "피해 굴림": "Damage Roll", "피해 임계값": "Damage Threshold",
    "피해 유형": "Damage Types", "어둠": "Darkness", "암시야": "Darkvision", "질주": "Dash", "사망": "Dead",
    "죽음 내성 굴림": "Death Saving Throw", "탈수": "Dehydration", "험지": "Difficult Terrain", "난이도 등급": "Difficulty Class",
    "희미한 빛": "Dim Light", "이탈": "Disengage", "회피": "Dodge", "발산": "Emanation", "조우": "Encounter", "적": "Enemy",
    "경험치": "Experience Points", "낙하": "Falling", "비행 속도": "Fly Speed", "비행": "Flying", "우호적": "Friendly",
    "위험 요소": "Hazard", "치유": "Healing", "심하게 가려짐": "Heavily Obscured", "도움": "Help", "영웅적 영감": "Heroic Inspiration",
    "은신": "Hide", "높이뛰기": "High Jump", "체력 주사위": "Hit Point Dice", "체력": "Hit Points", "적대적": "Hostile",
    "부양": "Hover", "면역": "Immunity", "임시변통 무기": "Improvised Weapons", "무관심": "Indifferent", "영향": "Influence",
    "도약": "Jumping", "약간 가려짐": "Lightly Obscured", "직선": "Line", "멀리뛰기": "Long Jump", "마법": "Magic",
    "마법 효과": "Magical Effect", "영양실조": "Malnutrition", "몬스터": "Monster", "비플레이어 캐릭터": "Nonplayer Character",
    "물체": "Object", "점유 공간": "Occupied Space", "수동 감지": "Passive Perception", "하루당": "Per Day",
    "플레이어 캐릭터": "Player Character", "소유": "Possession", "도달 거리": "Reach", "대기": "Ready", "의식": "Ritual",
    "내림": "Round Down", "내성": "Save", "수색": "Search", "변신": "Shape-Shifting", "동시 효과": "Simultaneous Effects",
    "크기": "Size", "기술": "Skill", "속도": "Speed", "주문": "Spell", "주문 공격": "Spell Attack", "주문 시전 매개체": "Spellcasting Focus",
    "구": "Sphere", "안정": "Stable", "스탯 블록": "Stat Block", "연구": "Study", "질식": "Suffocation", "수영 속도": "Swim Speed",
    "수영": "Swimming", "대상": "Target", "텔레파시": "Telepathy", "순간이동": "Teleportation", "임시 체력": "Temporary Hit Points",
    "진동감지": "Tremorsense", "진시": "Truesight", "맨손 타격": "Unarmed Strike", "빈 공간": "Unoccupied Space", "무기": "Weapon",
    "무기 공격": "Weapon Attack",
}

TERM_MAP = dict(TERM_V1)
print("용어표 v1:", len(TERM_MAP))

# 프롬프트에 넣을 글: "Advantage=이점, Disadvantage=불리점, ..."
TERMS = ", ".join(f"{en}={ko}" for ko, en in TERM_MAP.items())

# 거꾸로 찾기용: 영어(소문자) → 한국어
TERM_REV = {en.lower(): ko for ko, en in TERM_MAP.items()}


def terms_for(text):                # 근거 글에 나오는 용어만 골라 "영어=한국어" 목록 만들기
    out = []
    for ko, en in TERM_MAP.items():
        if " " in en:
            # 두 단어 이상 (예: Attack Roll): 복수형 s 허용, 대소문자 무시
            pattern = rf"\b{re.escape(en)}s?\b"     # 앞뒤에 \b를 붙이면 단어 전체가 일치할 때만 찾음
            found = re.search(pattern, text, re.I)
        else:
            # 한 단어 (예: Dash): 대소문자까지 정확히 (dash 같은 일반 단어와 구분)
            pattern = rf"\b{re.escape(en)}\b"
            found = re.search(pattern, text, 0)
        if found:
            out.append(f"{en}={ko}")
    return ", ".join(out) or "(없음)"      # 하나도 없으면 "(없음)"

# 전체 제목에 대한 용어표 사용 검토 -> 의미가 있나? --> 코드 내 삽입이 아닌 파일로 저장

In [ ]:
# Ex09-b. 호출 함수 + 깨짐/혼합/분수 검사

# 시스템 프롬프트: 한국어로만 답하고, 한자·일본어 등은 쓰지 말 것
SYSTEM = ("You are a D&D rules assistant. Write every answer in Korean (Hangul). "
          "Never use Chinese characters or Chinese words. "
          "Also never use Japanese, Vietnamese, Russian, Thai, or any other language or script. "
          "Exceptions: English D&D terms in parentheses only when the instructions ask for them, "
          "and the English source quote when asked to copy it.")


def _chat(prompt, model=None, system=SYSTEM, temperature=0.0, seed=GEN_SEED, max_tokens=None):   # LLM 호출 (기본)
    options = {"num_ctx": GEN_CTX, "temperature": temperature}
    if seed is not None:
        options["seed"] = seed                     # seed가 있으면 같은 결과로 고정
    if max_tokens:
        options["num_predict"] = max_tokens        # 출력 길이 제한 (짧은 답용)

    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or GEN_MODEL,
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": prompt}],
              "stream": False,
              "options": options},
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]


def ask_explain(prompt, model=None, temperature=0.1, seed=GEN_SEED):   # 설명 쓰기용 호출
    return _chat(prompt, model, SYSTEM, temperature, seed)


def ask_short(prompt, model=None, system=SYSTEM):    # 짧은 답용 호출 (번호, 예/아니오) — 최대 20토큰
    return _chat(prompt, model, system, 0.0, GEN_SEED, max_tokens=20)


# ── 출력 검사용 패턴 ──
# 이 글자들 중 하나라도 있으면 깨짐
#   一-鿿 한자 / ぀-ヿ 일본어 / Ạ-ỹ 및 ăâêôơưđ 베트남어
#   ̀-ͯ 글자 위에 붙는 기호 / � 깨진 글자(�)
BROKEN = re.compile(r"[一-鿿぀-ヿẠ-ỹ̀-ͯăâêôơưđĐ�]")


def is_broken(text):                # 깨진 글자가 있는지
    return bool(BROKEN.search(text))


LEAK = re.compile(r"(^|\n)\s*(user|assistant|system)\s*(\n|:)", re.I)
# 줄 처음에 "user" 같은 말이 새어 나왔는지

MIXED = re.compile(r"[가-힣][A-Za-z]")         # 한글 바로 뒤에 영어 철자가 붙음 (예: "이점Advantage", "스티일")
FRACTION = re.compile(r"\d+분의\s*\d+")        # 분수 뒤집는 오류 잡기


def bad_output(text):               # 다시 써야 하는 출력인지 (깨짐 / 혼합 철자 / 분수)
    if is_broken(text):
        return True
    if MIXED.search(text):
        return True
    if FRACTION.search(text):
        return True
    return False

# 서버 오류 → try/except + 재시도 1~2번 \
# 컨텍스트 → GEN_CTX 8192를 4096으로 변경 (빨라지지만 출력이 달라질 수 있어 개발셋 확인 필요)
# 스트리밍 → stream: True 변경 (서버 붙일 때, 나중)

In [ ]:
# Ex09-c. 검사 도구

def expl_part(text):                # 답변에서 "설명:" 뒤의 글만 꺼내기 (없으면 전체)
    m = re.search(r"설명\s*[:：]\s*(.*)", text, flags=re.S)   # re.S = 줄바꿈까지 포함해서 끝까지
    if m:
        return m.group(1).strip()
    return text


NUM = re.compile(r"\d+(?:d\d+)?", re.I)            # 숫자 또는 주사위 표기 (예: 30, 2d6)


def numbers(text):                  # 글에 나온 숫자·주사위 표기 목록 (소문자)
    return [n.lower() for n in NUM.findall(text)]


# 영어 숫자 단어 → 숫자 (근거에 "two"라고 적혀 있어도 설명의 "2"를 인정하려고)
W2N = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7",
       "eight": "8", "nine": "9", "ten": "10", "half": "1 2", "quarter": "1 4", "quarters": "4"}
W2N.update({"twice": "2", "double": "2", "doubled": "2"})


def norm_txt(s):                    # 비교용으로 글 다듬기 (따옴표 제거, 공백 하나로, 소문자)
    s = re.sub(r"[\"'“”‘’]", "", s)                # 따옴표 종류 전부 지우기
    s = re.sub(r"\s+", " ", s)                     # 공백·줄바꿈 여러 개 → 공백 하나
    return s.strip().lower()

## 10. 답변 파이프라인 (선택 → 검증/거절 → 설명 → 자동 검사)

```
질문 → retrieve(5개) → select_chunk(번호) → verify_chunk(예/아니오)
        └ 아니오 → 거절 ─┐
        └ 예 → explain_chunk(원문+설명) → check_explain(코드 검사) → check_faithful(근거에 없는 내용?)
                                      └ 걸리면: 개발 모드 = 숨기고 근거 본문 표시 / 챗봇 모드 = 다른 seed로 재시도 → 그래도 걸리면 "확실하지 않아요"
```


In [ ]:
# Ex10-a. 근거 선택: 후보 5개 중 번호 하나 고르기
SELECT_TEMPLATE = """다음은 D&D 규칙 질문과 후보 근거입니다. 질문에 직접 답하는 근거의 번호 하나만 고르세요.
- 질문이 묻는 규칙 자체를 설명하는 근거를 고르세요. 이름만 비슷하거나 예시로 지나가듯 언급되는 근거는 고르지 마세요.
- 숫자 하나만 출력하세요.

### 질문
{question}

### 후보 근거
{cands}

### 번호"""


def select_chunk(q, found, model=None):    # 후보 중 질문에 맞는 근거 번호 고르기 (1부터)
    # 후보마다 "[번호] 제목: 본문 앞부분 300자" 한 줄씩
    lines = []
    for n, c in enumerate(found, 1):
        preview = c["text"][:SELECT_CHARS].replace("\n", " ")   # 줄바꿈 → 공백 (한 줄로)
        lines.append(f"[{n}] {c['meta']['title']}: {preview}")
    cands = "\n".join(lines)

    out = ask_short(SELECT_TEMPLATE.format(question=q, cands=cands), model)

    # 답에서 첫 숫자 꺼내기. 숫자가 없거나 범위 밖이면 1번
    m = re.search(r"\d+", out)
    n = int(m.group()) if m else 1
    if 1 <= n <= len(found):
        return n
    return 1

In [ ]:
# Ex10-b. 설명하기 : 챗봇이 답을 쓰고, 코드가 그 답을 확인

# 설명 프롬프트
EXPLAIN_TEMPLATE = """당신은 D&D 규칙 안내자입니다. 아래 [근거] 하나만 사용해 질문에 답하세요.
1. 근거에 적힌 내용만 사용하세요. 숫자, 조건, 이점/불리점을 빼거나 바꾸지 마세요.
2. 용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적어. 같은 용어는 항상 같은 한국어로 써: {terms}
   '이점'은 Advantage에만 쓰고, bonus는 '보너스', roll은 '굴림', check는 '판정'으로 써.
3. 답변은 "원문:", "설명:" 두 줄로 쓰세요. 원문은 질문에 직접 답하는 문장 1~3개를 영어 그대로, 끝까지, 한 줄로 복사하세요.
   설명은 근거에 있는 내용만 한국어로 쓰세요. 질문이 묻는 내용은 빠짐없이 쓰고(목록이면 항목 전부, 이점/불리점은 누구의 굴림인지 구분), 묻지 않은 다른 규칙이나 예시는 쓰지 마세요.
   인사, 서론, 되묻기는 쓰지 마세요. "원문:"과 "설명:"은 각각 한 번만 쓰세요.

### 근거
{context}

### 질문
{question}

### 답변"""

# 2번 규칙을 바꿔 끼우기용: 영어 괄호 표기 있음(EN) / 없음(KO, 챗봇 화면용)
RULE_EN = "용어는 이 표의 한국어를 쓰고 영어 원문을 괄호로 함께 적어. 같은 용어는 항상 같은 한국어로 써: {terms}"
RULE_KO = "용어는 이 표의 한국어만 쓰세요. 영어는 괄호로도 적지 마세요. 같은 용어는 항상 같은 한국어로 써: {terms}"
assert RULE_EN in EXPLAIN_TEMPLATE, "2번 규칙 문구가 달라요"


def explain_chunk(question, k=TOP_K, tries=EXPLAIN_TRIES, temp0=0.1, seed0=40, pre=None, en_terms=True):   # 근거 하나로 원문+설명 쓰기
    # pre = (검색 결과, 검색어, 고른 번호)를 미리 받으면 검색·선택을 다시 안 해요 (answer에서 사용)
    # seed0 / temp0 = 챗봇 재시도 때 바꿔 쓰는 값 (기본값이면 이전과 같아요)

    # 1. 검색 + 번호 고르기
    if pre:
        found, texts, n = pre[0], pre[1], pre[2]
    else:
        found, texts = retrieve(question, k)
        n = select_chunk(question, found)
    c = found[n - 1]                                   # 고른 근거 (번호는 1부터라 -1)

    # 2. 프롬프트 만들기
    ctx = f"({c['meta']['title']} p.{c['meta']['page']}) {c['text'][:CTX_CHARS]}"
    if en_terms:
        tmpl = EXPLAIN_TEMPLATE
    else:
        tmpl = EXPLAIN_TEMPLATE.replace(RULE_EN, RULE_KO)
    prompt = tmpl.format(terms=terms_for(c["text"]), context=ctx, question=question)

    # 3. 설명 쓰기: 깨지면 온도·seed를 바꿔 최대 tries번
    for t in range(1, tries + 1):
        temperature = temp0 + 0.3 * (t - 1)           # 1번째 0.1 → 2번째 0.4 → 3번째 0.7
        out = ask_explain(prompt, temperature=temperature, seed=seed0 + t)
        if not bad_output(out):
            break

    # 4. "원문:" 꺼내서 근거에 진짜 있는 문장인지 확인
    m = re.search(r"원문\s*[:：]\s*(.+)", out)
    orig = m.group(1).strip() if m else ""
    verified = bool(orig) and norm_txt(orig) in norm_txt(c["text"])

    first_line, _, rest = c["text"].partition("\n")    # 청크 첫 줄 = 경로, 나머지 = 본문
    body = rest if "\n" in c["text"] else first_line

    return {
        "title": c["meta"]["title"],
        "page": c["meta"]["page"],
        "text": c["text"],
        "orig_ok": verified,
        "orig": orig if verified else body[:300],      # 검증 실패면 본문 앞 300자로 대신
        "path": first_line,
        "explain": expl_part(out),
        "picked": n,                                   # 고른 번호
        "queries": texts[1:],                          # 번역
        "tries": t,                                    # 설명을 몇 번 만에 썼는지
    }


def check_explain(r):               # 설명을 코드로 검사 (숫자·형식·철자 등)
    ok = r["orig_ok"]

    # 원문이 문장 끝(. ! ? ")으로 끝나는지
    complete = ok and r["orig"].rstrip().endswith((".", "!", "?", '"'))

    # 근거에 있는 숫자 모음: 숫자 그대로 + 영어 숫자 단어(two → 2)
    src = set(numbers(r["text"]))
    for w in re.findall(r"[a-z]+", r["text"].lower()):
        if w in W2N:
            for d in W2N[w].split():                   # "half" → "1 2" → {"1", "2"}
                src.add(d)

    # 근거에 없는 숫자 찾기: 설명에 나온 숫자를 하나씩 보고, 근거에 없으면 모아 두기
    bad_nums = []
    for num in numbers(r["explain"]):      # 설명에 나온 숫자들 (예: "30", "2d6")
        if num not in src:                 # src = 근거에 나온 숫자 모음
            bad_nums.append(num)           # 근거에 없음 → 지어낸 숫자일 수 있음

    # 원문에 나온 용어인데 설명에 한국어가 없는 것 (원문 검증 실패면 검사 안 함)
    term_miss = []
    if ok:
        for ko, en in TERM_MAP.items():
            if re.search(rf"\b{re.escape(en)}\b", r["orig"]) and ko not in r["explain"]:
                term_miss.append(ko)

    return {
        "원문_검증": ok,
        "원문_완결": complete,
        "근거에_없는_숫자": bad_nums,
        "용어_누락": term_miss,
        "혼합_철자": bool(MIXED.search(r["explain"])),
        "분수": FRACTION.findall(r["explain"]),
        "형식_누출": bool(re.search(r"(원문|설명)\s*[:：]", r["explain"])),   # 설명 안에 "원문:"이 또 나옴
    }

In [ ]:
# Ex10-c. 근거 검증 · 충실성 검사 · 최종 함수 `answer` / `show_answer`

# ── 1. 근거 검증: 고른 근거가 질문에 진짜 답이 되는지 LLM에게 예/아니오로 묻기 ──
VERIFY_TEMPLATE = """다음 D&D 규칙 질문과 근거입니다. 이 근거가 질문에 직접 답하는 규칙 내용을 담고 있으면 '예', 이름만 비슷하거나 다른 주제이면 '아니오'라고만 답하세요.
질문은 한국어, 근거는 영어예요. 표현이 달라도 같은 규칙을 다루면 '예'예요.

### 질문
{question}
(영어 검색어: {en})

### 근거
{title}: {text}

### 답 (예/아니오)"""


def verify_chunk(q, c, en=""):      # 근거가 질문에 맞는지 LLM에게 묻기 (예 → True)
    prompt = VERIFY_TEMPLATE.format(question=q, en=en or "-", title=c["meta"]["title"],
                                    text=c["text"][:VERIFY_CHARS])
    out = ask_short(prompt)
    out = re.sub(r"[\s\*'\"]", "", out)               # 공백·별표·따옴표 지우기 ("**예**" → "예")
    return out.startswith(("예", "네"))


# ── 2. 충실성 검사: 설명에 근거에 없는 내용이 있는지 LLM에게 한 번 더 묻기 ──
FAITH_TEMPLATE = """다음은 D&D 규칙 근거(영어)와 그것을 한국어로 설명한 글입니다.
설명에 근거에 없는 내용(지어낸 사실, 근거와 다른 숫자·조건)이 있으면 '있음', 없으면 '없음'이라고만 답하세요.

### 근거
{src}

### 설명
{exp}

### 답 (있음/없음)"""


def check_faithful(r):              # 설명이 근거대로 썼는지 검사 (True = 통과)
    src = r["text"][:CTX_CHARS]     # 근거 원문 (앞 2000자)
    exp = r["explain"]              # 챗봇이 쓴 설명

    prompt = FAITH_TEMPLATE.format(src=src, exp=exp)        # format() : 문자열 안의 {이름}을 값으로 바꿔 넣는 기능
    reply = ask_short(prompt).strip()      # LLM 답: "없음" 또는 "있음"

    if reply.startswith("없음"):    # 지어낸 내용 없음 → 통과
        return True
    return False                    # "있음"이거나 엉뚱한 답 → 걸림


# ── 3. 경고 모으기 ──
def collect_warnings(r, chk, check_faith=True, show_orig=True):   # 검사 결과 → 경고 목록 (빈 목록 = 통과)
    w = []
    if not chk["원문_검증"]:
        w.append("원문 검증 실패")
    elif show_orig and not chk["원문_완결"]:          # 원문을 보여 줄 때만 끊김을 따짐
        w.append("원문이 문장 중간에서 끊김")
    if chk["근거에_없는_숫자"]:
        w.append(f"근거에 없는 숫자 {chk['근거에_없는_숫자']}")
    if chk["분수"]:
        w.append(f"분수 표현 {chk['분수']}")
    if chk["형식_누출"]:
        w.append("형식 누출")
    if chk["혼합_철자"]:
        w.append("혼합 철자")
    if is_broken(r["explain"]):
        w.append("깨진 글자")

    # 코드 검사를 다 통과했을 때만 LLM 충실성 검사 (느려서 마지막에)
    if not w and check_faith and not check_faithful(r):
        w.append("근거에 없는 내용")
    return w


# ── 4. 검증 생략 조건 ──
def sure_match(texts, c):           # 근거 제목이 번역1,2 안에 그대로 있는지 (있으면 확실)
    t = norm(c["meta"]["title"])
    for x in texts[1:]:                                # texts = [원문, 번역①, 번역②]
        if re.search(rf"\b{re.escape(t)}\b", x.lower()):
            return True
    return False


def passes_verify(q, c, texts):     # 근거 통과 여부 (제목이 맞으면 바로 통과, 아니면 LLM 검증)
    return sure_match(texts, c) or verify_chunk(q, c, texts[1])


# ── 5. 최종 함수 ──
def answer(question, attempts=1, show_orig=True):    # 질문 → 검색 → 선택 → 검증 → 설명 → 검사
    # 검색 + 번호 고르기
    found, texts = retrieve(question)
    n = select_chunk(question, found)
    c = found[n - 1]

    # 검증 실패 → 거절
    if not passes_verify(question, c, texts):

        # 후보 3개의 (제목, 페이지)만 모아 두기 → 개발 화면에서 "후보: ..."로 보여 줌
        candidates = []
        for x in found[:3]:
            title = x["meta"]["title"]
            page = x["meta"]["page"]
            candidates.append((title, page))

        return {
            "status": "refuse",          # 거절했다는 표시
            "candidates": candidates,    # 후보 3개
            "warn": [],                  # 경고 없음 (설명을 안 썼으니까)
            "first_flagged": False,      # 통계용: 첫 시도에 검사에 걸렸는지
            "attempts_used": 0,          # 통계용: 설명을 몇 번 썼는지 (거절이라 0)
        }

    # 설명 쓰기 + 검사: 경고가 있으면 seed를 바꿔 최대 attempts번
    first_flagged = None                               # 첫 시도에 경고가 있었는지 (통계용)
    r, warn = None, []
    for a in range(attempts):
        temp0 = 0.0 if a == 0 else 0.3                 # 첫 시도는 0, 재시도부터 0.3
        r = explain_chunk(question, temp0=temp0, seed0=40 + 10 * a,
                          pre=(found, texts, n), en_terms=show_orig)
        warn = collect_warnings(r, check_explain(r), show_orig=show_orig)
        if first_flagged is None:
            first_flagged = bool(warn)
        if not warn:
            break

    first_line, _, rest = r["text"].partition("\n")
    body = rest if "\n" in r["text"] else first_line   # 청크에서 경로(첫 줄)를 뺀 본문

    return {
        "status": "hidden" if warn else "ok",          # 경고가 남아 있으면 설명을 숨김
        "warn": warn,
        "first_flagged": first_flagged,
        "attempts_used": a + 1,
        "title": r["title"],
        "page": r["page"],
        "path": r["path"],
        "orig": r["orig"],
        "body": body[:600],
        "explain": r["explain"],
    }


def show_answer(question, dev_view=None):    # 답변을 화면에 출력 (개발 화면 / 챗봇 화면)
    if dev_view is None:
        dev_view = DEV_VIEW

    # 개발 화면: 1번만 시도 + 원문 표시 / 챗봇 화면: 최대 CHAT_ATTEMPTS번 + 설명만
    if dev_view:
        r = answer(question, attempts=1, show_orig=True)
    else:
        r = answer(question, attempts=CHAT_ATTEMPTS, show_orig=False)

    if r["status"] == "refuse":                        # 거절
        if dev_view:
            print("🙅 이 질문에 맞는 규칙 근거를 확실히 찾지 못했어요." + f" 후보: {r['candidates']}")
        else:
            print("🙅 이 질문에 맞는 규칙 근거를 확실히 찾지 못했어요." + " 질문을 바꿔 보시겠어요?")

    elif r["status"] == "hidden":                      # 검사에 걸려서 숨김
        if dev_view:
            print(f"⚠️ 설명을 숨겼어요 {r['warn']}\n출처: {r['title']} p.{r['page']} ({r['path']})\n[근거 본문] {r['body']}")
        else:
            print("확실하지 않아요. 질문을 조금 바꿔서 다시 물어봐 주세요.")

    else:                                              # 정상
        if dev_view:
            print(f"출처: {r['title']} p.{r['page']} ({r['path']})\n원문: {r['orig']}\n설명: {r['explain']}")
        else:
            print(r["explain"])
    return r

### 질문 해 보기 (여기만 바꿔서 다시 실행)
`DEV_VIEW = True`면 원문·페이지·근거 본문까지, `False`면 챗봇처럼 설명만 보여 줘요. 캐시에 없는 새 질문은 모델 교체 때문에 첫 질문이 20초쯤 걸릴 수 있어요.

In [ ]:
question = "넘어졌을 때 일어나려면 이동력이 얼마나 들어?"
_ = show_answer(question)

## 📝 실험 기록

**검색**
- 번역을 7B → 14B로 바꿔 개발셋 검색을 다시 쟀다 → 66문항 합계 (43, 52, 59)로 7B (41, 54, 57)와 비슷.
- 번역 방식 A~E 비교 → 세 갈래 RRF(E)가 가장 좋아서 채택.
- 키워드(BM25) 합치기 비교 → 키워드 비중 0.5(F2)를 검색 v1로 채택.
- 번역①의 용어 힌트를 빼 봤다 → 점수가 비슷해서 힌트 없는 일반 번역으로 확정.
- 한국어 + 번역 두 번 검색해서 합치기 → 이득 없어서 기각.
- 재순위(reranker) 실험 → 1등이 오히려 줄어서(31 → 28) 기각. -> 재시도
- Prone·Hide 질문은 어떤 방식으로도 잘 못 찾음 → 알려진 약점. -> 더 있을지도?

**답변**
- 근거 선택을 "한 번에 하기" vs "2단계(번호 먼저)"로 비교 → 2단계 채택.
- 근거 검증(예/아니오) 평가 + 거절 정책 비교 → 정책 A(아니오면 바로 거절) 채택.
- 설명 충실성 검사(근거에 없는 내용 묻기)를 추가했다.
- 7B·14B 설명에서 한자 섞임, 분수 뒤집힘(3분의 4) 발견 → 깨짐·혼합·분수 검사와 재생성으로 이어짐.
- 용어표 주입 켬/끔 비교(38문항) → 켬: 정상 31·숨김 3·용어누락 9 / 끔: 정상 28·숨김 6·용어누락 18 → 켬 유지.
- 모델 교체(14B ↔ bge-m3)가 느림의 원인 → 번역·검색 캐시를 만들었다.
- temperature 0 + seed 42로 같은 결과가 나오는지 확인 → 재현됨.

**최종 세트 (golden_final, 60문항)**
- v1 점수는 31 / 46 / 51. 한 번만 재는 시험지라 결과 파일이 있으면 다시 못 재게 잠가 뒀었다.